# Exploratório — Diagnóstico das bases (Correios × Geosanja)

**Objetivo:** olhar as duas bases *antes* de cruzar qualquer coisa, para fechar as regras de padronização.
Este notebook **não altera nem cruza nada**: só lê, conta e mostra amostras.

**Estrutura**
1. Configuração (caminhos, log)
2. Funções de diagnóstico
3. Diagnóstico dos Correios (planilha `.xlsx`)
4. Diagnóstico do Geosanja — tabela de atributos
5. Diagnóstico do Geosanja — geometria
6. Resumo das conferências

> Ao terminar, copie o **log das células 4, 5, 6 e 7** e cole na conversa.

## 1. Configuração
Define os caminhos dos arquivos e liga o log (mensagens com data/hora no console).
Mensagens terminadas em `...` indicam que a etapa **começou**; terminadas em `!` indicam que **terminou**.

In [ ]:
import logging                      # módulo padrão do Python para registrar mensagens (melhor que print)
from pathlib import Path             # forma moderna de trabalhar com caminhos de arquivos
import pandas as pd                  # tabelas
import pyogrio                       # leitura rápida de arquivos geográficos (.gpkg)

# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "exploratorio_diagnostico_bases"    # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/

# --- Caminhos dos arquivos --------------------------------------------------
ARQ_CORREIOS = cfg.caminho("brutos.correios")
ARQ_GEOSANJA = cfg.caminho("brutos.geosanja")

# --- Valores esperados (usados só para conferir) ---------------------------
ESPERADO_CORREIOS_LINHAS = 5330      # linhas de dados da planilha dos Correios
ESPERADO_GEOSANJA_FEICOES = 9435     # feições da camada no QGIS
ESPERADO_CRS = 31983                 # SIRGAS 2000 / UTM 23S

# --- Abreviações de títulos que vamos procurar nos nomes do Geosanja -------
ABREVIACOES = {
    "DR", "DRA", "PROF", "PROFA", "CEL", "ENG", "ENGO", "MAJ", "CAP", "PE", "STO", "STA",
    "GEN", "TEN", "SGT", "ALM", "MAL", "DEP", "PRES", "VER", "GOV", "MONS", "CMT", "BRIG",
}

CONFERENCIAS = {}                    # guarda o resultado (True/False) de cada conferência para o resumo final

log.info("Configuração carregada!")
for nome, caminho in [("Correios", ARQ_CORREIOS), ("Geosanja", ARQ_GEOSANJA)]:
    log.info("%s: %s | existe: %s", nome, caminho.name, caminho.exists())

## 2. Funções de diagnóstico
Só definem as funções (nada é lido ainda). Cada uma recebe uma tabela e **escreve no log** o que encontrou.

In [ ]:
def normalizar_basico(serie: pd.Series) -> pd.Series:
    """Padronização mínima só para o diagnóstico: MAIÚSCULAS, sem acento, sem pontuação, sem espaços extras.

    (A padronização completa do pareamento será feita na próxima etapa.)
    """
    s = serie.fillna("").astype(str)
    s = s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")  # tira acentos
    s = s.str.upper()                                                              # caixa alta
    s = s.str.replace(r"[^A-Z0-9 ]", " ", regex=True)                              # troca pontuação por espaço
    return s.str.replace(r"\s+", " ", regex=True).str.strip()                      # tira espaços duplicados


def limpar_texto(serie: pd.Series) -> pd.Series:
    """Remove espaços nas pontas e transforma texto vazio em nulo (NA)."""
    s = serie.astype("string").str.strip()
    return s.mask(s == "", pd.NA)


def mostrar(titulo: str, tabela, max_linhas: int = 40) -> None:
    """Escreve um título e uma tabela no log, limitando o número de linhas."""
    if isinstance(tabela, pd.Series):
        tabela = tabela.to_frame()
    log.info("%s (mostrando até %d de %d linhas):\n%s", titulo, max_linhas, len(tabela),
             tabela.head(max_linhas).to_string())


def conferir(nome: str, ok: bool, detalhe: str) -> None:
    """Registra uma conferência: OK ou ALERTA, e guarda o resultado para o resumo final."""
    CONFERENCIAS[nome] = ok
    (log.info if ok else log.warning)("[%s] %s — %s", "OK" if ok else "ALERTA", nome, detalhe)


# ---------------------------------------------------------------------------
# CORREIOS
# ---------------------------------------------------------------------------
def carregar_correios(caminho: Path) -> pd.DataFrame:
    """Lê a planilha dos Correios (aba 'Page 1') com tudo como texto, para não perder zeros."""
    log.info("Lendo planilha dos Correios...")
    df = pd.read_excel(caminho, sheet_name="Page 1", dtype=str, keep_default_na=False)
    log.info("Planilha dos Correios lida: %d linhas x %d colunas!", *df.shape)
    return df


def diagnosticar_correios(df: pd.DataFrame) -> None:
    """Confere a planilha dos Correios: tamanho, CEP, situação e tipos de logradouro."""
    log.info("Diagnosticando Correios...")
    conferir("Correios: nº de linhas", len(df) == ESPERADO_CORREIOS_LINHAS,
             f"{len(df)} lidas, esperado {ESPERADO_CORREIOS_LINHAS}")

    cep = df["CEP"].str.strip()
    formato_ok = cep.str.fullmatch(r"\d{5}-\d{3}")                 # formato 12345-678
    conferir("Correios: formato do CEP", bool(formato_ok.all()), f"{int((~formato_ok).sum())} fora do padrão 00000-000")
    log.info("CEPs distintos: %d | linhas com CEP repetido: %d", cep.nunique(), int(cep.duplicated(keep=False).sum()))

    mostrar("Situação (coluna 4)", df["Situação"].value_counts())
    mostrar("Tipo de codificação", df["Tipo Codificação"].value_counts())

    tipo = df["Logradouro"].str.strip().str.split().str[0]          # primeira palavra = tipo (Rua, Avenida...)
    mostrar("Primeira palavra do logradouro (tipo)", tipo.value_counts())
    log.info("Diagnóstico dos Correios concluído!")


# ---------------------------------------------------------------------------
# GEOSANJA — atributos
# ---------------------------------------------------------------------------
def carregar_geosanja_atributos(caminho: Path):
    """Lê só a tabela de atributos do .gpkg (sem geometria, bem mais rápido). Devolve (tabela, nome_da_camada, info)."""
    log.info("Listando camadas do GeoPackage...")
    camadas = pyogrio.list_layers(caminho)                          # lista [nome, tipo de geometria]
    log.info("Camadas encontradas: %s", [(str(c[0]), str(c[1])) for c in camadas])
    camada = str(camadas[0][0])                                          # usa a primeira (esperado: só existe uma)
    if len(camadas) > 1:
        log.warning("Há mais de uma camada; usando '%s'. Confirme se é a correta!", camada)

    info = pyogrio.read_info(caminho, layer=camada)                 # CRS, nº de feições, campos
    log.info("Lendo atributos da camada '%s'...", camada)
    df = pyogrio.read_dataframe(caminho, layer=camada, read_geometry=False)
    log.info("Atributos lidos: %d linhas x %d colunas!", *df.shape)
    return df, camada, info


def diagnosticar_geosanja_atributos(df: pd.DataFrame, info: dict | None = None) -> None:
    """Diagnostica a tabela de atributos do Geosanja: nulos, 'Sem nome', tipos, abreviações, alfanuméricos e duplicados."""
    log.info("Diagnosticando atributos do Geosanja...")

    # --- 1) Tamanho, CRS e chave ------------------------------------------
    conferir("Geosanja: nº de feições", len(df) == ESPERADO_GEOSANJA_FEICOES,
             f"{len(df)} lidas, esperado {ESPERADO_GEOSANJA_FEICOES}")
    if info is not None:
        crs = str(info.get("crs"))
        conferir("Geosanja: CRS", str(ESPERADO_CRS) in crs, f"CRS do arquivo: {crs}")
    conferir("Geosanja: 'id' único e sem nulos", bool(df["id"].notna().all() and df["id"].is_unique),
             f"{int(df['id'].isna().sum())} nulos, {int(df['id'].duplicated().sum())} repetidos")

    # --- 2) Colunas de texto limpas (strip + vazio vira nulo) --------------
    for col in ["nome", "tipo", "loteamento", "nome_ante", "diploma", "numero", "ano"]:
        df[col] = limpar_texto(df[col])
    nulos = pd.DataFrame({"preenchidos": df[["nome", "tipo", "loteamento", "nome_ante", "diploma", "numero", "ano"]].notna().sum()})
    nulos["% preenchido"] = (nulos["preenchidos"] / len(df) * 100).round(1)
    mostrar("Preenchimento das colunas de texto", nulos)

    # --- 3) Nome padronizado e vias 'sem nome' -----------------------------
    nome_norm = normalizar_basico(df["nome"])
    sem_nome = nome_norm.isin(["", "SEM NOME"])
    log.info("Vias SEM NOME (nulo, vazio ou 'Sem nome'): %d (%.1f%%)", sem_nome.sum(), sem_nome.mean() * 100)
    log.info("  ...dessas, com loteamento preenchido: %d", int((sem_nome & df["loteamento"].notna()).sum()))
    log.info("Vias COM nome mas SEM tipo: %d", int((~sem_nome & df["tipo"].isna()).sum()))
    log.info("Vias SEM nome mas COM tipo: %d", int((sem_nome & df["tipo"].notna()).sum()))
    validas = df[~sem_nome].copy()
    validas["nome_norm"] = nome_norm[~sem_nome]
    validas["tipo_norm"] = normalizar_basico(validas["tipo"])

    # --- 4) Tipos de logradouro --------------------------------------------
    mostrar("Tipos (vias com nome)", validas["tipo_norm"].replace("", "(VAZIO)").value_counts())

    # --- 5) Primeiras palavras do nome (para achar títulos) ----------------
    primeira = validas["nome_norm"].str.split().str[0]
    mostrar("Primeira palavra do nome — 40 mais comuns", primeira.value_counts(), 40)

    # --- 6) Abreviações de títulos e iniciais ------------------------------
    tokens = validas["nome_norm"].str.split().explode()
    abrev = tokens[tokens.isin(ABREVIACOES)].value_counts()
    log.info("Abreviações de título encontradas (DR, PROF, CEL...): %d ocorrências", int(abrev.sum()))
    if len(abrev):
        mostrar("Abreviações encontradas", abrev)
    com_ponto = validas.loc[validas["nome"].str.contains(r"\.", na=False), "nome"]
    log.info("Nomes com ponto (ex.: iniciais 'M.'): %d", len(com_ponto))
    if len(com_ponto):
        mostrar("Amostra de nomes com ponto", com_ponto.drop_duplicates())
    uma_letra = validas.loc[validas["nome_norm"].str.contains(r"(?:^| )[A-Z](?: |$)", na=False), "nome"]
    log.info("Nomes com palavra de uma só letra (inicial ou 'Rua D'): %d", len(uma_letra))
    if len(uma_letra):
        mostrar("Amostra de nomes com letra isolada", uma_letra.drop_duplicates())

    # --- 7) Alfanuméricos e caracteres especiais ---------------------------
    com_digito = validas.loc[validas["nome_norm"].str.contains(r"\d", na=False), "nome"]
    log.info("Nomes com dígitos (ex.: H10B, 31, Km): %d", len(com_digito))
    if len(com_digito):
        mostrar("Amostra de nomes com dígitos", com_digito.drop_duplicates(), 40)
    especiais = validas.loc[validas["nome"].str.contains(r"[-/()'\"ºª&,]", na=False), "nome"]
    log.info("Nomes com hífen, barra, parênteses, aspas, º/ª, & ou vírgula: %d", len(especiais))
    if len(especiais):
        mostrar("Amostra de nomes com caracteres especiais", especiais.drop_duplicates(), 30)

    # --- 8) Nomes repetidos (segmentos da mesma rua) ------------------------
    validas["chave"] = (validas["tipo_norm"] + " " + validas["nome_norm"]).str.strip()
    grupo = validas.groupby("chave").agg(segmentos=("id", "size"), loteamentos=("loteamento", "nunique"))
    log.info("Logradouros distintos (tipo + nome): %d | com mais de 1 segmento: %d | com >1 loteamento: %d",
             len(grupo), int((grupo["segmentos"] > 1).sum()), int((grupo["loteamentos"] > 1).sum()))
    mostrar("Nomes mais repetidos", grupo.sort_values("segmentos", ascending=False), 20)

    # --- 9) Loteamento, nome anterior e demais campos ----------------------
    log.info("Loteamentos distintos: %d | vias com nome e SEM loteamento: %d",
             validas["loteamento"].nunique(), int(validas["loteamento"].isna().sum()))
    mostrar("Loteamentos mais frequentes", validas["loteamento"].value_counts(), 15)
    ante = validas.loc[validas["nome_ante"].notna(), ["nome", "nome_ante"]]
    log.info("Vias com nome anterior preenchido: %d", len(ante))
    if len(ante):
        mostrar("Amostra de nome anterior", ante, 15)
    for col in ["diploma", "numero", "ano"]:
        log.info("Campo '%s' preenchido em %d vias", col, int(df[col].notna().sum()))

    log.info("Diagnóstico dos atributos do Geosanja concluído!")


# ---------------------------------------------------------------------------
# GEOSANJA — geometria
# ---------------------------------------------------------------------------
def diagnosticar_geometria(caminho: Path, camada: str) -> None:
    """Confere a geometria: vazias, inválidas, tipos, comprimento total e extensão (bounding box)."""
    log.info("Lendo geometrias do Geosanja...")
    gdf = pyogrio.read_dataframe(caminho, layer=camada, columns=["id"])   # só o id + geometria
    log.info("Geometrias lidas: %d feições!", len(gdf))

    vazias = int(gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum())
    invalidas = int((~gdf.geometry.is_valid).sum())
    conferir("Geometrias vazias/nulas", vazias == 0, f"{vazias} encontradas")
    conferir("Geometrias inválidas", invalidas == 0, f"{invalidas} encontradas")
    mostrar("Tipos de geometria", gdf.geom_type.value_counts())
    log.info("CRS da camada: %s", gdf.crs)
    log.info("Comprimento total da malha: %.1f km", gdf.length.sum() / 1000)
    log.info("Extensão (xmin, ymin, xmax, ymax): %s", [round(float(v), 1) for v in gdf.total_bounds])
    log.info("Diagnóstico da geometria concluído!")


log.info("Funções definidas!")

## 3. Diagnóstico dos Correios
Lê a planilha `.xlsx` e confere tamanho, formato do CEP, situação e tipos de logradouro.
(Já analisei o CSV equivalente; aqui só confirmo que a planilha lida é a mesma.)

In [ ]:
df_correios = carregar_correios(ARQ_CORREIOS)
diagnosticar_correios(df_correios)

## 4. Diagnóstico do Geosanja — tabela de atributos
Lê só os atributos do `.gpkg` e mostra: vias sem nome, tipos, abreviações, formato dos alfanuméricos, nomes repetidos, loteamentos.

In [ ]:
df_geo, CAMADA_GEO, info_geo = carregar_geosanja_atributos(ARQ_GEOSANJA)
diagnosticar_geosanja_atributos(df_geo, info_geo)

## 5. Diagnóstico do Geosanja — geometria
Lê as geometrias (só o `id` + linha) e confere se há feições vazias ou inválidas.

In [ ]:
diagnosticar_geometria(ARQ_GEOSANJA, CAMADA_GEO)

## 6. Resumo das conferências
Lista todas as conferências feitas acima. Qualquer `ALERTA` deve ser olhado antes de seguir.

In [ ]:
log.info("Resumo das conferências...")
for nome, ok in CONFERENCIAS.items():
    log.info("  [%s] %s", "OK" if ok else "ALERTA", nome)
n_alertas = sum(not ok for ok in CONFERENCIAS.values())
(log.info if n_alertas == 0 else log.warning)("Total: %d conferências, %d alerta(s)!", len(CONFERENCIAS), n_alertas)